In [ ]:
# ==========================
# Sakha NER mBERT
# ==========================

import os, sys
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

!pip -q install transformers datasets seqeval evaluate huggingface_hub

from google.colab import drive
drive.mount("/content/drive")

import json, random, hashlib, subprocess, shlex
from pathlib import Path
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from datasets import Dataset, concatenate_datasets
from evaluate import load
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, EarlyStoppingCallback

import torch, torch.nn as nn

# ==========================
# Paths & constants
# ==========================
MAIN_DIR      = "/content/drive/My Drive/"
DATA_JSON     = f"{MAIN_DIR}/sakha-ner-merged_with_negatives.json"
FAIR_SPLIT    = f"{MAIN_DIR}/sakha_ner_docsplit_FAIR_v1.json"

BASE_REPO_ID  = "bert-base-multilingual-cased"
BASE_CACHE    = f"{MAIN_DIR}/hf_cache/bert-base-multilingual-cased"

EXT_DIR = f"{MAIN_DIR}/mbert_extended_wp_sakha_pieceavg_fair"

# Training hyperparams
SEED=42
MAX_LEN=256
TRAIN_STRIDE=32
EVAL_STRIDE=0
ORG_MULT=2
LR=3e-5
EPOCHS=8
TRAIN_BS=8
GRAD_ACC=4
EVAL_BS=16
WD=0.01
WARMUP=0.1

LABELS   = ["O","B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

# ==========================
# Utilities
# ==========================
def ensure_cached_model(repo_id: str, local_dir: str):
    """Download a hub model once into local_dir if missing; training stays offline afterwards."""
    p = Path(local_dir) / "config.json"
    if p.exists():
        print("📦 Base model already cached:", local_dir)
        return
    prev = os.environ.get("HF_HUB_OFFLINE", "1")
    os.environ["HF_HUB_OFFLINE"] = "0"
    for m in [k for k in list(sys.modules) if k.startswith("huggingface_hub")]:
        del sys.modules[m]
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=repo_id, local_dir=local_dir, local_dir_use_symlinks=False, resume_download=True)
        print("✅ Downloaded base to:", local_dir)
    except Exception as e:
        raise SystemExit(f"❌ Could not download {repo_id}. Reason:\n{e}\nProvide it at {local_dir} and rerun.")
    finally:
        os.environ["HF_HUB_OFFLINE"] = prev

def load_ls(p):
    with open(p, "r", encoding="utf-8") as f:
        s=f.read(1); f.seek(0)
        return json.load(f) if s=="[" else [json.loads(line) for line in f if line.strip()]

def last_annotation_results(task):
    anns = task.get("annotations", [])
    if not anns: return None
    last = anns[-1]
    if last.get("was_cancelled", False): return None
    out, txt = [], task["data"]["text"]; n = len(txt)
    for r in last.get("result", []):
        v=r.get("value", {}); labs=v.get("labels", [])
        if labs and "start" in v and "end" in v:
            s,e=int(v["start"]),int(v["end"])
            if 0<=s<e<=n: out.append((s,e,labs[0]))
    return out

def get_doc_key(task):
    d = task.get("data", {})
    for k in ("file_upload","document_id","doc_id","source","file"):
        if k in d and d[k]:
            return f"{k}:{str(d[k])}"
    return "sha1:" + hashlib.sha1(d.get("text","").encode("utf-8")).hexdigest()

# ==========================
# Determinism & GPU info
# ==========================
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False

print("Deterministic:", torch.are_deterministic_algorithms_enabled())
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0), "| CC:", torch.cuda.get_device_capability(0))
    try: print(subprocess.check_output(shlex.split("nvidia-smi")).decode("utf-8"))
    except: pass

tasks_all = [t for t in load_ls(DATA_JSON) if last_annotation_results(t) is not None]
docs = defaultdict(list)
for t in tasks_all:
    docs[get_doc_key(t)].append(t)
doc_ids = list(docs.keys())

def doc_has_ent(dk): return int(any(last_annotation_results(x) for x in docs[dk]))
y = np.array([doc_has_ent(dk) for dk in doc_ids])

if Path(FAIR_SPLIT).exists():
    print("Using existing fair split:", FAIR_SPLIT)
    with open(FAIR_SPLIT, "r", encoding="utf-8") as f:
        sp = json.load(f)
    train_doc_keys, dev_doc_keys, test_doc_keys = sp["train_doc_keys"], sp["dev_doc_keys"], sp["test_doc_keys"]
else:
    tr, tmp, y_tr, y_tmp = train_test_split(doc_ids, y, test_size=0.30, random_state=SEED, stratify=y)
    dv, te, y_dv, y_te   = train_test_split(tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
    train_doc_keys, dev_doc_keys, test_doc_keys = tr, dv, te
    with open(FAIR_SPLIT, "w", encoding="utf-8") as f:
        json.dump({"train_doc_keys": tr, "dev_doc_keys": dv, "test_doc_keys": te}, f, ensure_ascii=False, indent=2)
    print("Wrote fair split to:", FAIR_SPLIT)

def _ratio(a,b): return 0 if b==0 else round(a/b,3)
print("Doc split sizes:",
      {"train": len(train_doc_keys), "dev": len(dev_doc_keys), "test": len(test_doc_keys)})
print("Doc-level neg rate:",
      {split: _ratio(sum(1-doc_has_ent(dk) for dk in ks), len(ks))
       for split, ks in [("train", train_doc_keys), ("dev", dev_doc_keys), ("test", test_doc_keys)]})

def windowed_features(task, tokenizer, max_len=MAX_LEN, stride=TRAIN_STRIDE):
    txt = task["data"]["text"]
    spans = last_annotation_results(task) or []
    enc = tokenizer(txt, return_offsets_mapping=True, return_overflowing_tokens=True,
                    truncation=True, max_length=max_len, stride=stride,
                    padding="max_length", pad_to_multiple_of=8)
    feats=[]; specials=set(tokenizer.all_special_ids)
    for i in range(len(enc["input_ids"])):
        ids = enc["input_ids"][i]; attn = enc["attention_mask"][i]; offs = enc["offset_mapping"][i]
        labels=[-100 if tid in specials else label2id["O"] for tid in ids]
        votes={}
        for (s,e,lab) in spans:
            touched=[]
            for j,(a,b) in enumerate(offs):
                if labels[j]==-100: continue
                ov=max(0, min(b,e)-max(a,s))
                if ov>0: touched.append((j,ov))
            if not touched: continue
            touched.sort(key=lambda x:x[0])
            for k,(j,ov) in enumerate(touched):
                tag=("B-" if k==0 else "I-")+lab
                cand=(label2id[tag], ov)
                if (j not in votes) or (ov>votes[j][1]): votes[j]=cand
        for j,c in votes.items(): labels[j]=c[0]
        prev="O"
        for j in range(len(labels)):
            if labels[j]==-100: continue
            tag=LABELS[labels[j]]
            if tag.startswith("I-"):
                ent=tag[2:]
                if prev not in (f"B-{ent}", f"I-{ent}"):
                    labels[j]=label2id[f"B-{ent}"]; tag=LABELS[labels[j]]
            prev=tag
        feats.append({"input_ids": ids, "attention_mask": attn, "labels": labels})
    return feats

def make_ds(doc_keys, tokenizer, stride):
    def gen():
        for dk in doc_keys:
            for t in docs[dk]:
                for w in windowed_features(t, tokenizer, MAX_LEN, stride):
                    yield w
    return Dataset.from_generator(gen)

metric = load("seqeval")
def compute_metrics(eval_pred):
    preds = np.asarray(eval_pred.predictions); labs = eval_pred.label_ids
    pid = preds.argmax(-1)
    true_labels = [[LABELS[int(l)] for l in lab if l!=-100] for lab in labs]
    true_preds  = [[LABELS[int(p)] for p,l in zip(p_seq, lab) if l!=-100] for p_seq,lab in zip(pid,labs)]
    res = metric.compute(predictions=true_preds, references=true_labels)
    out = {
        "precision": res.get("overall_precision",0.0),
        "recall":    res.get("overall_recall",0.0),
        "f1":        res.get("overall_f1",0.0),
        "accuracy":  res.get("overall_accuracy",0.0),
    }
    for ent,v in res.items():
        if isinstance(v,dict):
            for m in ("precision","recall","f1","number"):
                if m in v: out[f"{m}_{ent}"]=v[m]
    return out

class EncoderLinearHead(nn.Module):
    def __init__(self, base_dir, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_dir, local_files_only=True)
        h = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(h, num_labels))
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):
        kw.pop("num_items_in_batch", None)
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kw)
        logits = self.classifier(out.last_hidden_state)
        loss=None
        if labels is not None:
            loss = nn.CrossEntropyLoss(ignore_index=-100)(logits.view(-1, logits.size(-1)), labels.view(-1))
        return {"loss": loss, "logits": logits}

def run_arm(model_dir: str, tag: str):
    tok = AutoTokenizer.from_pretrained(model_dir, use_fast=True, local_files_only=True)
    emb_n = AutoModel.from_pretrained(model_dir, local_files_only=True).get_input_embeddings().weight.size(0)
    assert len(tok)==emb_n, f"Vocab/embedding mismatch: tok={len(tok)} vs emb={emb_n}"

    train_ds = make_ds(train_doc_keys, tok, TRAIN_STRIDE)
    dev_ds   = make_ds(dev_doc_keys,   tok, EVAL_STRIDE)
    test_ds  = make_ds(test_doc_keys,  tok, EVAL_STRIDE)

    if ORG_MULT>1:
        def has_org(ex): return any(y in (label2id["B-ORG"], label2id["I-ORG"]) for y in ex["labels"])
        org_subset = train_ds.filter(has_org)
        train_ds = concatenate_datasets([train_ds] + [org_subset for _ in range(ORG_MULT-1)])
        print(f"[{tag}] ORG oversampling: +{len(org_subset)*(ORG_MULT-1)} (x{ORG_MULT})")
    print(f"[{tag}] windows -> train={len(train_ds)} dev={len(dev_ds)} test={len(test_ds)}")

    # training args
    ts = __import__("datetime").datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = f"{MAIN_DIR}/ner_runs_mbert/{tag}_{ts}"
    os.makedirs(run_dir, exist_ok=True)
    args = TrainingArguments(
        output_dir=run_dir,
        eval_strategy="epoch",
        learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=WARMUP,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BS, gradient_accumulation_steps=GRAD_ACC,
        per_device_eval_batch_size=EVAL_BS,
        weight_decay=WD,
        logging_dir=f"{run_dir}/logs", logging_steps=25,
        save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
        fp16=False, bf16=False, tf32=False, dataloader_num_workers=2, report_to="none",
        seed=SEED, max_grad_norm=1.0, group_by_length=True,
    )

    model = EncoderLinearHead(model_dir, num_labels=len(LABELS), dropout=0.1)
    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds, eval_dataset=dev_ds,
                      tokenizer=tok, compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    print(f"\n[{tag}] Starting training (backbone: {model_dir}) …")
    trainer.train()
    print(f"[{tag}] Best ckpt:", trainer.state.best_model_checkpoint)

    test_metrics = trainer.evaluate(test_ds)

    # pretty per-label
    def pl(lbl):
        return (test_metrics.get(f"eval_f1_{lbl}"), test_metrics.get(f"eval_precision_{lbl}"),
                test_metrics.get(f"eval_recall_{lbl}"), test_metrics.get(f"eval_number_{lbl}"))
    print(f"\n[{tag}] === TEST per-label ===")
    for ent in ("LOC","ORG","PER"):
        f1,p,r,n = pl(ent)
        print(f"{ent:>3} | F1={f1:.3f}  P={p:.3f}  R={r:.3f}  N={int(n)}")
    print(f"[{tag}] OVERALL: F1={test_metrics.get('eval_f1'):.3f} | Acc={test_metrics.get('eval_accuracy'):.3f}\n")

    best_dir = f"{run_dir}/best_model"
    trainer.save_model(best_dir)
    tok.save_pretrained(best_dir)
    with open(f"{run_dir}/metrics_TEST.json","w",encoding="utf-8") as f:
        json.dump(test_metrics, f, ensure_ascii=False, indent=2)

    pred = trainer.predict(test_ds)
    logits = pred.predictions
    pid = np.asarray(logits).argmax(-1) if isinstance(logits, np.ndarray) and logits.ndim==3 else np.asarray(logits)
    lab_ids = pred.label_ids
    preds_iob, golds_iob = [], []
    for p, l in zip(pid, lab_ids):
        keep = l != -100
        preds_iob.append([LABELS[int(x)] for x in p[keep]])
        golds_iob.append([LABELS[int(x)] for x in l[keep]])
    with open(f"{run_dir}/predictions_TEST.jsonl", "w", encoding="utf-8") as f:
        for pseq, gseq in zip(preds_iob, golds_iob):
            f.write(json.dumps({"pred": pseq, "gold": gseq}, ensure_ascii=False) + "\n")

    cfg = {
        "base_model": model_dir,
        "labels": LABELS,
        "vocab_size": int(len(tok)),
        "emb_rows": int(AutoModel.from_pretrained(model_dir, local_files_only=True)
                        .get_input_embeddings().weight.size(0)),
        "max_len": MAX_LEN, "train_stride": TRAIN_STRIDE, "eval_stride": EVAL_STRIDE,
        "org_mult": ORG_MULT, "epochs": EPOCHS, "lr": LR, "bs": TRAIN_BS,
        "grad_accum": GRAD_ACC, "seed": SEED, "split_file": FAIR_SPLIT,
        "final_f1": float(test_metrics.get("eval_f1", 0.0)),
        "best_checkpoint": trainer.state.best_model_checkpoint,
    }
    with open(f"{run_dir}/config.json","w",encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)

    print(f"✅ [{tag}] saved to: {run_dir}")
    return run_dir, test_metrics

ensure_cached_model(BASE_REPO_ID, BASE_CACHE)
os.environ["HF_HUB_OFFLINE"] = "1"

ARMS = [
    ("base_mbert",        BASE_CACHE),
    ("extended_mbert",    EXT_DIR),
]

results = {}
for tag, model_dir in ARMS:
    if tag.startswith("extended") and not Path(model_dir).exists():
        raise SystemExit(f"❌ Extended checkpoint not found at: {model_dir}")
    run_dir, metrics = run_arm(model_dir, tag)
    results[tag] = {"run_dir": run_dir, "f1": metrics.get("eval_f1"),
                    "f1_LOC": metrics.get("eval_f1_LOC"),
                    "f1_ORG": metrics.get("eval_f1_ORG"),
                    "f1_PER": metrics.get("eval_f1_PER")}

print("\n=== SUMMARY mBERT ===")
for tag in ARMS:
    nm = tag[0]
    r  = results[nm]
    print(f"{nm:>16}: F1={r['f1']:.3f} | LOC={r['f1_LOC']:.3f} ORG={r['f1_ORG']:.3f} PER={r['f1_PER']:.3f}  -> {r['run_dir']}")


In [ ]:
tok = AutoTokenizer.from_pretrained(EXT_DIR)
model = AutoModel.from_pretrained(EXT_DIR)
print(f"Tokenizer vocab: {len(tok)}")
print(f"Model embeddings: {model.get_input_embeddings().weight.shape[0]}")

In [ ]:
model = AutoModel.from_pretrained(EXT_DIR)
emb = model.get_input_embeddings().weight
print(f"Original embeddings std: {emb[:119547].std()}")
print(f"New embeddings std: {emb[119547:].std()}")